# 07 — Gradio Demo

Interactive before/after demo comparing zero-shot vs fine-tuned Cosmos-Reason2-2B
on underwater scenes. Loads the QLoRA adapter from Drive.

**Run order:** Cell 1 → Cell 2 → Cell 3 → Cell 4 → Cell 5

- Cell 1: Install
- Cell 2: Mount Drive + login + load videos
- Cell 3: Load fine-tuned model
- Cell 4: Inference helpers
- Cell 5: Launch Gradio

## Cell 1 — Install

In [ ]:
!pip install -Uq gradio decord fiftyone peft

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.2/24.2 MB 133.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.3/58.3 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 159.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.8/11.8 MB 137.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.8/112.8 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.6/61.6 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.4/112.4 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 74.8/74.8 kB 7.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 212.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 153.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.4/316.4 kB 43.2 MB/s eta 

## Cell 2 — Mount Drive, Login & Load Videos

In [ ]:
from google.colab import drive, userdata
from huggingface_hub import login

drive.mount('/content/drive', force_remount=True)
login(token=userdata.get('HF_TOKEN'))

import re, json, os, torch
import numpy as np
from PIL import Image
import decord
import fiftyone as fo
from fiftyone.utils.huggingface import load_from_hub

# Define resolve_video function (missing)
def resolve_video(path):
    # Paths from fiftyone are typically absolute and directly usable.
    # This function can be expanded if more complex path resolution is needed.
    return path

# ── Config ─────────────────────────────────────────────
MODEL_NAME   = 'nvidia/Cosmos-Reason2-2B'
ADAPTER_DIR  = '/content/drive/MyDrive/cosmos-cookoff/outputs/final_model_distilled'
QUAL_PATH    = '/content/drive/MyDrive/cosmos-cookoff/outputs/evaluation/qualitative_comparison.json'
N_FRAMES     = 8   # 4 frames for demo speed (~30-40s per inference)

# Must be identical to notebooks 03, 05, 06
SYSTEM_PROMPT = (
    'You are an expert underwater robotics vision system for AUV navigation. '
    'Analyze the underwater video frames and respond ONLY with a valid JSON object. '
    'You MUST output all 9 fields as separate keys. '
    'Do NOT put everything inside scene_assessment. '
    'Each field has a specific type and purpose:\n'
    '  scene_assessment: 1-2 sentence description of the environment\n'
    '  visibility_level: exactly one of HIGH, MEDIUM, LOW, CRITICAL\n'
    '  visibility_gt_meters: numeric estimate of visibility range in meters\n'
    '  primary_subject: what the main subject is, where it is, how it moves\n'
    '  hazards: JSON array of strings, each describing one navigation hazard\n'
    '  hazard_level: exactly one of NONE, LOW, MEDIUM, HIGH, CRITICAL\n'
    '  recommended_action: exactly one of PROCEED, REDUCE_SPEED, STOP, TURN_LEFT, TURN_RIGHT, ASCEND, DESCEND, ABORT_MISSION\n'
    '  action_reasoning: one sentence explaining why that action was chosen\n'
    '  confidence: float between 0.0 and 1.0\n\n'
    'Example of correct output format:\n'
    '{"scene_assessment": "Coastal sea environment with moderate blue-green water.", '
    '"visibility_level": "MEDIUM", '
    '"visibility_gt_meters": 4.0, '
    '"primary_subject": "Sea turtle, center-right of frame, moving left", '
    '"hazards": ["Partial occlusion by coral structure"], '
    '"hazard_level": "LOW", '
    '"recommended_action": "PROCEED", '
    '"action_reasoning": "Path is clear with only minor visual obstructions.", '
    '"confidence": 0.78}'
    '\n\nOutput ONLY the JSON object. No markdown fences, no explanation.'
)

# ── Discover video paths ─────────────────────────────────────
print('Loading WebUOT-238-Test via fiftyone...')
try:
    fo_dataset = load_from_hub('Voxel51/WebUOT-238-Test', overwrite=False)
except ValueError:
    # Dataset already exists in local fiftyone — just load it directly
    print('Dataset already cached, loading from local fiftyone store...')
    fo_dataset = fo.load_dataset('Voxel51/WebUOT-238-Test')

path_lookup = {os.path.basename(s.filepath): s.filepath for s in fo_dataset}
print(f'Videos available: {len(path_lookup)}')

# ── Load qualitative examples from notebook 06 ──────────────────
PRESET_VIDEOS = []
VAL_PATH = '/content/drive/MyDrive/cosmos-cookoff/data/underwater_vqa_val_unified.json'

if os.path.exists(QUAL_PATH) and os.path.exists(VAL_PATH):
    with open(QUAL_PATH) as f:
        qual = json.load(f)
    with open(VAL_PATH) as f:
        val_data = json.load(f)

    # Build id → video path lookup from val set
    id_to_video = {r['id']: r['video'] for r in val_data}

    for r in qual:
        stored_path = id_to_video.get(r['id'])
        if not stored_path:
            print(f"  No video found for {r['id']}")
            continue
        resolved = resolve_video(stored_path)
        if os.path.exists(resolved):
            PRESET_VIDEOS.append({
                'label':     r['question_type'].replace('_', ' ').title(),
                'video':     resolved,
                'gt_action': r.get('gt_action', 'N/A'),
            })
    print(f'Loaded {len(PRESET_VIDEOS)} preset demo videos from notebook 06')
else:
    print('qualitative_comparison.json or val JSON not found — upload your own video in the demo')

print('\nCell 2 complete.')

Mounted at /content/drive
Loading WebUOT-238-Test via fiftyone...


INFO:fiftyone.utils.huggingface:Downloading config file fiftyone.yml from Voxel51/WebUOT-238-Test


Loading dataset


INFO:fiftyone.utils.huggingface:Loading dataset


Dataset already cached, loading from local fiftyone store...
Videos available: 238
Loaded 5 preset demo videos from notebook 06

Cell 2 complete.


## Cell 3 — Load Fine-Tuned Model

Loads the QLoRA adapter from Drive. Use `enable_adapter_layers()` /
`disable_adapter_layers()` to toggle between fine-tuned and zero-shot mode.

In [ ]:
from transformers import AutoProcessor, Qwen3VLForConditionalGeneration
from peft import PeftModel

print('Loading processor...')
processor = AutoProcessor.from_pretrained(MODEL_NAME, trust_remote_code=True)
processor.image_processor.min_pixels = 256 * 28 * 28
processor.image_processor.max_pixels = 512 * 28 * 28

print('Loading base model (Cosmos-Reason2-2B)...')
base_model = Qwen3VLForConditionalGeneration.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.bfloat16,
    device_map='auto',
    trust_remote_code=True,
)

print(f'Attaching QLoRA adapter from {ADAPTER_DIR}...')
ft_model = PeftModel.from_pretrained(base_model, ADAPTER_DIR)
ft_model.eval()
print(f'Model ready. VRAM: {torch.cuda.memory_allocated()/1e9:.1f} GB')

Loading processor...


preprocessor_config.json:   0%|          | 0.00/390 [00:00<?, ?B/s]

chat_template.json:   0%|          | 0.00/5.50k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.50k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/10.9k [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/4.88G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/626 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie model.language_model.embed_tokens.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/269 [00:00<?, ?B/s]

Attaching QLoRA adapter from /content/drive/MyDrive/cosmos-cookoff/outputs/final_model_distilled...
Model ready. VRAM: 5.0 GB


## Cell 4 — Inference & Formatting Helpers

In [ ]:
def extract_frames(video_path, n_frames=N_FRAMES):
    vr      = decord.VideoReader(video_path, ctx=decord.cpu(0))
    indices = np.linspace(0, len(vr) - 1, n_frames, dtype=int)
    frames  = vr.get_batch(indices).asnumpy()
    return [Image.fromarray(frames[i]) for i in range(n_frames)]


def parse_json_response(text):
    text = text.strip()
    if text.startswith('```'):
        text = text.split('```')[1]
        if text.startswith('json'): text = text[4:]
        text = text.strip()
    text = re.sub(r'"\s*\n(\s*")', '",\n\\1', text)
    try:
        return json.loads(text), True
    except Exception:
        pass
    start, end = text.find('{'), text.rfind('}')
    if start != -1 and end > start:
        try:
            return json.loads(text[start:end+1]), True
        except Exception:
            pass
    return {}, False


def run_inference(video_path, use_adapter=True, max_new_tokens=512):
    try:
        frames = extract_frames(video_path)
    except Exception as e:
        return None, f'[Frame extraction error: {e}]', False

    if use_adapter:
        ft_model.enable_adapter_layers()
    else:
        ft_model.disable_adapter_layers()

    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user',   'content': [
            *[{'type': 'image', 'image': f} for f in frames],
            {'type': 'text',   'text': 'Analyze this underwater scene and provide a structured navigation assessment for AUV control.'},
        ]},
    ]
    text   = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[text], images=frames, return_tensors='pt', padding=True)
    inputs = {k: v.to(ft_model.device) for k, v in inputs.items()}

    with torch.no_grad():
        out_ids = ft_model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)

    generated = out_ids[0][inputs['input_ids'].shape[1]:]
    raw = processor.decode(generated, skip_special_tokens=True).strip()
    parsed, ok = parse_json_response(raw)
    return parsed, raw, ok


HAZARD_COLORS = {
    'NONE':     ('#e8f5e9', '#1b5e20'),
    'LOW':      ('#e8f5e9', '#1b5e20'),
    'MEDIUM':   ('#fff8e1', '#e65100'),
    'HIGH':     ('#ffebee', '#b71c1c'),
    'CRITICAL': ('#b71c1c', '#ffffff'),
}

ACTION_LABELS = {
    'PROCEED':       'PROCEED',
    'REDUCE_SPEED':  'REDUCE SPEED',
    'STOP':          'STOP',
    'TURN_LEFT':     'TURN LEFT',
    'TURN_RIGHT':    'TURN RIGHT',
    'ASCEND':        'ASCEND',
    'DESCEND':       'DESCEND',
    'ABORT_MISSION': 'ABORT MISSION',
}


def format_result_html(parsed, ok, label):
    """Render the JSON result as clean HTML. All colors explicit — no theme inheritance."""
    if not ok or not parsed:
        return (
            '<div style="font-family:Arial,sans-serif;padding:16px;'
            'background:#fff3f3;border:1px solid #e57373;border-radius:8px;'
            'color:#b71c1c;">' + f'<b>{label}</b><br>Could not parse model response.' + '</div>'
        )

    action  = parsed.get('recommended_action', 'UNKNOWN')
    hazard  = parsed.get('hazard_level', 'UNKNOWN')
    vis     = parsed.get('visibility_level', 'UNKNOWN')
    vis_m   = parsed.get('visibility_gt_meters', '?')
    scene   = parsed.get('scene_assessment', '')
    subject = parsed.get('primary_subject', '')
    hazards = parsed.get('hazards', [])
    reason  = parsed.get('action_reasoning', '')

    try:
        vis_m = round(float(str(vis_m)), 1)
    except (ValueError, TypeError):
        vis_m = '?'

    try:
        conf_pct = int(float(str(parsed.get('confidence', 0))) * 100)
    except (ValueError, TypeError):
        conf_pct = 0

    hbg, hfg = HAZARD_COLORS.get(hazard, ('#f5f5f5', '#212121'))
    action_label = ACTION_LABELS.get(action, action)

    if hazards and isinstance(hazards, list):
        hazard_text = ' / '.join(str(h) for h in hazards if h)
    else:
        hazard_text = 'None detected'

    return f'''
<div style="font-family:Arial,sans-serif;background:#ffffff;border:1px solid #dee2e6;
            border-radius:8px;overflow:hidden;color:#212121;">

  <div style="background:#1c1c2e;color:#ffffff;padding:10px 14px;font-size:13px;font-weight:bold;">
    {label}
  </div>

  <div style="padding:14px 16px;background:#ffffff;">

    <div style="display:flex;gap:8px;margin-bottom:14px;">
      <div style="flex:1;background:{hbg};border:1px solid #ccc;border-radius:6px;
                  padding:10px 6px;text-align:center;">
        <div style="font-size:11px;color:{hfg};margin-bottom:3px;font-weight:bold;
                    letter-spacing:0.5px;">AUV ACTION</div>
        <div style="font-size:15px;font-weight:bold;color:{hfg};">{action_label}</div>
      </div>
      <div style="flex:1;background:{hbg};border:1px solid #ccc;border-radius:6px;
                  padding:10px 6px;text-align:center;">
        <div style="font-size:11px;color:{hfg};margin-bottom:3px;font-weight:bold;
                    letter-spacing:0.5px;">HAZARD LEVEL</div>
        <div style="font-size:15px;font-weight:bold;color:{hfg};">{hazard}</div>
      </div>
      <div style="flex:1;background:#e3f2fd;border:1px solid #90caf9;border-radius:6px;
                  padding:10px 6px;text-align:center;">
        <div style="font-size:11px;color:#0d47a1;margin-bottom:3px;font-weight:bold;
                    letter-spacing:0.5px;">VISIBILITY</div>
        <div style="font-size:14px;font-weight:bold;color:#0d47a1;">{vis}</div>
        <div style="font-size:11px;color:#1565c0;">{vis_m} m</div>
      </div>
      <div style="flex:1;background:#f3e5f5;border:1px solid #ce93d8;border-radius:6px;
                  padding:10px 6px;text-align:center;">
        <div style="font-size:11px;color:#4a148c;margin-bottom:3px;font-weight:bold;
                    letter-spacing:0.5px;">CONFIDENCE</div>
        <div style="font-size:15px;font-weight:bold;color:#4a148c;">{conf_pct}%</div>
      </div>
    </div>

    <table style="width:100%;border-collapse:collapse;font-size:13px;">
      <tr>
        <td style="padding:6px 0;vertical-align:top;width:72px;">
          <span style="font-weight:bold;color:#374151;">Scene</span>
        </td>
        <td style="padding:6px 0;color:#374151;line-height:1.5;">{scene}</td>
      </tr>
      <tr style="border-top:1px solid #f0f0f0;">
        <td style="padding:6px 0;vertical-align:top;">
          <span style="font-weight:bold;color:#374151;">Subject</span>
        </td>
        <td style="padding:6px 0;color:#374151;line-height:1.5;">{subject}</td>
      </tr>
      <tr style="border-top:1px solid #f0f0f0;">
        <td style="padding:6px 0;vertical-align:top;">
          <span style="font-weight:bold;color:#374151;">Hazards</span>
        </td>
        <td style="padding:6px 0;color:#374151;">{hazard_text}</td>
      </tr>
      <tr style="border-top:1px solid #f0f0f0;">
        <td style="padding:6px 0;vertical-align:top;">
          <span style="font-weight:bold;color:#374151;">Reasoning</span>
        </td>
        <td style="padding:6px 0;color:#374151;line-height:1.5;
                   border-left:3px solid #9ca3af;padding-left:10px;">{reason}</td>
      </tr>
    </table>

  </div>
</div>
'''

print('Helpers ready.')


Helpers ready.


## Cell 5 — Launch Gradio Demo

Upload any video or pick one of the preset examples from notebook 06.
Runs both zero-shot and fine-tuned inference side by side.
Expected time per analysis: ~40-60 seconds (4 frames, H100).

In [ ]:
import gradio as gr
import shutil, tempfile, os

# ── Serve a Drive path by copying to /tmp so Gradio can stream it ─────────────
def stage_video(src_path):
    """Copy a Google Drive file to /tmp so Gradio's file server can reach it."""
    if src_path is None or not os.path.exists(src_path):
        return None
    ext = os.path.splitext(src_path)[1] or ".mp4"
    dst = tempfile.NamedTemporaryFile(suffix=ext, delete=False, dir="/tmp").name
    shutil.copy2(src_path, dst)
    return dst


# ── Analysis function ─────────────────────────────────────────────────────────
def analyze_video(video_path, stored_path, progress=gr.Progress()):
    """
    video_path  – value from gr.Video (may be a Gradio-managed tmp path or None)
    stored_path – value from gr.State  (the original Drive path we set on dropdown change)
    We prefer stored_path when video_path looks unreliable.
    """
    # Resolve the best available path
    path = None
    if stored_path and os.path.exists(stored_path):
        path = stored_path
    elif isinstance(video_path, str) and os.path.exists(video_path):
        path = video_path
    elif isinstance(video_path, dict):
        candidate = video_path.get("name") or video_path.get("path", "")
        if candidate and os.path.exists(candidate):
            path = candidate

    if path is None:
        err = '<div style="padding:16px;color:red;">Please upload or select a video first.</div>'
        return err, err, "", None

    progress(0.1, desc="Extracting frames...")

    progress(0.2, desc="Running zero-shot inference (no fine-tuning)...")
    zs_parsed, zs_raw, zs_ok = run_inference(path, use_adapter=False)

    progress(0.6, desc="Running fine-tuned inference...")
    ft_parsed, ft_raw, ft_ok = run_inference(path, use_adapter=True)

    progress(0.95, desc="Formatting results...")

    zs_html = format_result_html(zs_parsed, zs_ok, "Zero-Shot (base model)")
    ft_html = format_result_html(ft_parsed, ft_ok, "Fine-Tuned (Blue Recon)")

    delta_lines = []
    if zs_ok and ft_ok:
        for key, label in [("recommended_action", "Action"), ("hazard_level", "Hazard level"), ("visibility_level", "Visibility")]:
            zv, fv = zs_parsed.get(key, "N/A"), ft_parsed.get(key, "N/A")
            if zv != fv:
                delta_lines.append(f"**{label}:** {zv} → {fv}")
    if not zs_ok and ft_ok:
        delta_lines.append("**JSON parse:** FAILED → SUCCESS")

    delta_md = "### What changed after fine-tuning\n" + (
        "\n".join(f"- {l}" for l in delta_lines) if delta_lines
        else "- No structural differences (both parsed OK)"
    )

    # Clear stored_path after use so stale value doesn't leak to next upload
    return zs_html, ft_html, delta_md, None


# ── Preset loader: copies file to /tmp so gr.Video can serve it ───────────────
preset_choices = [(p["label"] + f" (GT: {p['gt_action']})", p["video"]) for p in PRESET_VIDEOS]
label_to_path  = {label: video for label, video in preset_choices}

def load_preset(choice):
    """Return (staged_video_for_preview, original_path_for_state)."""
    if not choice:
        return None, None
    original = label_to_path.get(choice)
    if not original or not os.path.exists(original):
        print(f"[load_preset] Path not found: {original}")
        return None, None
    staged = stage_video(original)
    return staged, original   # → video_input, path_state


def auto_analyze(staged_path, original_path, progress=gr.Progress()):
    """Called automatically when a preset is selected."""
    return analyze_video(staged_path, original_path, progress)


# ── Gradio UI ─────────────────────────────────────────────────────────────────
with gr.Blocks(title="Blue Recon — AUV Scene Understanding", theme=gr.themes.Ocean()) as demo:

    gr.Markdown("""
    # Blue Recon — Underwater AUV Scene Understanding
    **Cosmos-Reason2-2B** fine-tuned on WebUOT-238 for autonomous underwater vehicle navigation.

    Upload an underwater video **or** choose a preset — analysis runs automatically on preset selection.
    """)

    # Hidden state: holds the real Drive path so analyze_video is never path-blind
    path_state = gr.State(value=None)

    with gr.Row():
        with gr.Column(scale=1):
            video_input = gr.Video(label="Upload Underwater Video", height=280)

            if preset_choices:
                gr.Markdown("**Or choose a verified example from the evaluation set:**")
                preset_dd = gr.Dropdown(
                    choices=[label for label, _ in preset_choices],
                    label="Preset examples (ground truth action shown)",
                    interactive=True,
                    value=None,
                )

            analyze_btn = gr.Button("Analyze Scene", variant="primary", size="lg")

            gr.Markdown("""
            **Notes:**
            - ~40-60s per analysis (H100, 8 frames)
            - Both models use the same system prompt and question
            - Fine-tuned: QLoRA r=32 + 8B teacher distillation on 404 training samples
            """)

    with gr.Row():
        zs_out  = gr.HTML(label="Zero-Shot Output")
        ft_out  = gr.HTML(label="Fine-Tuned Output")

    delta_out = gr.Markdown(label="Delta Summary")

    # 1) Dropdown → stage video for preview AND store real path in State
    #    Then immediately run analysis (auto_analyze)
    if preset_choices:
        preset_dd.change(
            fn=load_preset,
            inputs=[preset_dd],
            outputs=[video_input, path_state],
        ).then(
            fn=auto_analyze,
            inputs=[video_input, path_state],
            outputs=[zs_out, ft_out, delta_out, path_state],
        )

    # 2) Manual upload clears the stored state (we rely on video_input directly)
    video_input.change(
        fn=lambda _: None,
        inputs=[video_input],
        outputs=[path_state],
    )

    # 3) Manual analyze button works for both uploaded and preset videos
    analyze_btn.click(
        fn=analyze_video,
        inputs=[video_input, path_state],
        outputs=[zs_out, ft_out, delta_out, path_state],
    )

    gr.Markdown("""
    ---
    ### Results Summary (30-sample evaluation)

    | Metric | Zero-Shot | Fine-Tuned | Delta |
    |---|---|---|---|
    | JSON parse rate | 73.3% | 96.7% | **+23.3pp** |
    | Action accuracy | 0.0% | 27.6% | **+27.6pp** |
    | Visibility accuracy | 0.0% | 93.1% | **+93.1pp** |
    | Avg fields / 9 | 6.9 | 8.7 | **+1.8** |

    *Zero-shot predicts PROCEED on 19/22 parseable samples regardless of scene content.*
    *Fine-tuned distributes across STOP / ABORT_MISSION / REDUCE_SPEED based on visual hazards.*
    """)

demo.launch(share=True, debug=False)
print("\nGradio demo is live.")


/tmp/ipykernel_7233/3186937092.py:147: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(title='Blue Recon — AUV Scene Understanding', theme=gr.themes.Ocean()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://b4c1aafad6e01539bc.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



Gradio demo is live. Share the public URL above.
